# Módulo 2 — Sentimiento: BETO Fine-tuning
**dccuchile/bert-base-spanish-wwm-uncased** con Hugging Face Trainer

> Activar GPU: Runtime → Change runtime type → T4 GPU

In [1]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/smartretail360'
MODEL_DIR  = f'{DRIVE_ROOT}/models/sentiment_analyzer/beto_finetuned'

import os
os.makedirs(MODEL_DIR, exist_ok=True)

Mounted at /content/drive


In [2]:
!pip install -q transformers datasets tensorflow scikit-learn

In [3]:
from huggingface_hub import list_repo_files, hf_hub_download
from datasets import load_dataset
from transformers import AutoTokenizer
import os

MODEL_NAME = 'dccuchile/bert-base-spanish-wwm-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

LANG = 'es'

# Descubrir y descargar solo los JSONL del español (el repo tiene un .py incompatible con datasets >= 4.0)
all_files  = list(list_repo_files("mteb/amazon_reviews_multi", repo_type="dataset"))
es_jsonl   = sorted([f for f in all_files if f.endswith('.jsonl') and f.startswith(f'{LANG}/')])

split_files = {}
for f in es_jsonl:
    local = hf_hub_download(
        repo_id="mteb/amazon_reviews_multi", filename=f,
        repo_type="dataset", local_dir="/content/amz_reviews_es",
    )
    fname = os.path.basename(f).lower()
    if 'train' in fname:   split_files.setdefault('train', []).append(local)
    elif 'test' in fname:  split_files.setdefault('test',  []).append(local)
    elif 'val'  in fname:  split_files.setdefault('validation', []).append(local)

ds = load_dataset("json", data_files=split_files)

def rating_to_label(r):
    # label va de 0 a 4 (estrellas -1), lo convertimos a 3 clases
    if r <= 1:   return 0  # negativo  (1-2 estrellas)
    elif r == 2: return 1  # neutro    (3 estrellas)
    else:        return 2  # positivo  (4-5 estrellas)

def tokenize_and_label(batch):
    out = tokenizer(batch['text'], truncation=True, max_length=256)
    out['labels'] = [rating_to_label(s) for s in batch['label']]
    return out

ds = ds.map(tokenize_and_label, batched=True, remove_columns=ds['train'].column_names)
print(ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/310 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/486k [00:00<?, ?B/s]

...

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5000
    })
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5000
    })
})


In [4]:
# Ver qué columnas tiene el dataset
print("Columnas disponibles:", ds['train'].column_names)

# Ver un ejemplo real del dataset
print("\nEjemplo de fila:")
print(ds['train'][0])

Columnas disponibles: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']

Ejemplo de fila:
{'input_ids': [4, 28850, 26909, 1150, 1505, 1491, 1057, 1094, 1247, 3097, 8289, 1035, 1885, 1009, 997, 2817, 1040, 1054, 1440, 5302, 3488, 1081, 13060, 5], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': 0}


In [5]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import f1_score, classification_report
import numpy as np

# Cargar modelo ya entrenado desde Drive (no entrenar de nuevo)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
print("Modelo cargado desde Drive correctamente")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo cargado desde Drive correctamente


In [6]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1_macro = f1_score(labels, predictions, average='macro')
    return {'f1_macro': f1_macro}

training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    per_device_eval_batch_size=32,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=ds['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\n=== Evaluación final ===")
preds_output = trainer.predict(ds['validation'])
preds = np.argmax(preds_output.predictions, axis=-1)
labels = preds_output.label_ids

print(f'F1-macro: {f1_score(labels, preds, average="macro"):.4f}')
print(classification_report(labels, preds, target_names=['negativo', 'neutro', 'positivo']))


=== Evaluación final ===


F1-macro: 0.7455
              precision    recall  f1-score   support

    negativo       0.84      0.85      0.85      2000
      neutro       0.52      0.50      0.51      1000
    positivo       0.88      0.88      0.88      2000

    accuracy                           0.79      5000
   macro avg       0.75      0.74      0.75      5000
weighted avg       0.79      0.79      0.79      5000



In [7]:
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print(f'BETO guardado en {MODEL_DIR}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BETO guardado en /content/drive/MyDrive/smartretail360/models/sentiment_analyzer/beto_finetuned
